# Nightly SQM Measurements
**GOAL:**
<br> Plot nightly SQM measurements:
- *x-axis* = Time [sunset -> sunrise]
- *y-axis* = SQM 'mags'
- Filtere the data by cloudcover, lunar phase.

## 1) Read the data

In [207]:
import sqm_plot202X
import numpy as np
import datetime
import pandas as pd
import sys
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import plotly.express as px
import plotly.graph_objects as go

In [208]:
# User inputs:
loc = 'SPZN' # Available data: FCZN, DSZN, HILT, SPZN
year = '2020'
month = '7'

In [209]:
# Load the CSV
fdir = '/Users/jacksontobin/Local_Documents/NightTime_Research/FoCo Night Sky Team/SQM_DATA/SQM_COMPLETE'
df_csv = f'{fdir}/{loc}_combined.csv'
df = pd.read_csv(df_csv)
df = df.set_index(df['date'])
df.index = pd.to_datetime(df.index)

# Remove bogus data
cond = (df['mags'] < 0)
df['mags'] = df['mags'].mask(cond, np.nan)

# Filter the date by years
df_yr = df[(df.index < datetime.datetime(year=int(year)+1, month=1, day=1)) & 
           (df.index > datetime.datetime(year=int(year),   month=1, day=1))]

# Filter out data if the solar elevation is > -18 (astronomical twighlight cutoff)
df_yr.loc[(df_yr['sunalt_deg']) >= -18, 'mags'] = float('nan')


In [210]:
# Choose the dates
df_mo = df_yr[df_yr.index.month.isin([int(month)])]

## 3) Sort data by day?

### Filter by Lunar and Cloud data

NOTE: 'date' column is a datetime object in the local time!

In [ ]:
# Convert the index to DateTime objects
df_mo.index = pd.to_datetime(df_mo.index)

# Convert to MST
# df_mo.index = df_mo.index.tz_localize('UTC').tz_convert('America/Denver')

,Unnamed: 0.1,date,mags,sunalt_deg,lunar_alt,lunar_az,lunar_fraction,lunar_phase,lunarphaseclass,Unnamed: 0,cloudcover,cloudcover_low,cloudcover_mid,cloudcover_high,CC
date,,,,,,,,,,,,,,,
2020-07-01 00:00:00,105549,2020-07-01 00:00:00,20.08,-24.228969,65.322160,204.544262,0.790016,0.365214,Gibbous,48.0,41.0,0.0,26.0,41.0,SCT
2020-07-01 00:01:00,105550,2020-07-01 00:01:00,20.05,-24.281309,65.556866,204.299168,0.790081,0.365214,Gibbous,48.0,41.0,0.0,26.0,41.0,SCT
2020-07-01 00:02:00,105551,2020-07-01 00:02:00,20.07,-24.332871,65.791449,204.054073,0.790147,0.365214,Gibbous,48.0,41.0,0.0,26.0,41.0,SCT
2020-07-01 00:03:00,105552,2020-07-01 00:03:00,20.06,-24.383652,66.025922,203.808965,0.790212,0.365214,Gibbous,48.0,41.0,0.0,26.0,41.0,SCT
2020-07-01 00:04:00,105553,2020-07-01 00:04:00,20.10,-24.433650,66.260266,203.563843,0.790278,0.365214,Gibbous,48.0,41.0,0.0,26.0,41.0,SCT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-07-31 23:55:00,122227,2020-07-31 23:55:00,19.31,-28.826771,40.131685,229.317891,0.922016,0.426347,Gibbous,792.0,37.0,0.0,37.0,24.0,SCT
2020-07-31 23:56:00,122228,2020-07-31 23:56:00,19.32,-28.889598,40.349421,229.074545,0.922056,0.426347,Gibbous,792.0,37.0,0.0,37.0,24.0,SCT
2020-07-31 23:57:00,122229,2020-07-31 23:57:00,19.32,-28.951610,40.567065,228.831172,0.922096,0.426347,Gibbous,792.0,37.0,0.0,37.0,24.0,SCT


In [212]:

# Create a new column: all 8pm-6am periods fall on the same date
df_mo['night'] = (df_mo.index - pd.Timedelta(hours=6)).normalize().tz_localize(None) # strips TZ for clean grouping
# df_mo['night'] = np.where(df_mo.index.hour >= 12, df_mo.index.date, (df_mo.index - pd.Timedelta(days=1)).date)  # Adjust for times after midnight
df_mo

/var/folders/hx/hcs55bp10f5fdvcmjf0rmf4r0000gn/T/ipykernel_70673/2111197200.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,Unnamed: 0.1,date,mags,sunalt_deg,lunar_alt,lunar_az,lunar_fraction,lunar_phase,lunarphaseclass,Unnamed: 0,cloudcover,cloudcover_low,cloudcover_mid,cloudcover_high,CC,night
date,,,,,,,,,,,,,,,,
2020-07-01 00:00:00,105549,2020-07-01 00:00:00,20.08,-24.228969,65.322160,204.544262,0.790016,0.365214,Gibbous,48.0,41.0,0.0,26.0,41.0,SCT,2020-06-30
2020-07-01 00:01:00,105550,2020-07-01 00:01:00,20.05,-24.281309,65.556866,204.299168,0.790081,0.365214,Gibbous,48.0,41.0,0.0,26.0,41.0,SCT,2020-06-30
2020-07-01 00:02:00,105551,2020-07-01 00:02:00,20.07,-24.332871,65.791449,204.054073,0.790147,0.365214,Gibbous,48.0,41.0,0.0,26.0,41.0,SCT,2020-06-30
2020-07-01 00:03:00,105552,2020-07-01 00:03:00,20.06,-24.383652,66.025922,203.808965,0.790212,0.365214,Gibbous,48.0,41.0,0.0,26.0,41.0,SCT,2020-06-30
2020-07-01 00:04:00,105553,2020-07-01 00:04:00,20.10,-24.433650,66.260266,203.563843,0.790278,0.365214,Gibbous,48.0,41.0,0.0,26.0,41.0,SCT,2020-06-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-07-31 23:55:00,122227,2020-07-31 23:55:00,19.31,-28.826771,40.131685,229.317891,0.922016,0.426347,Gibbous,792.0,37.0,0.0,37.0,24.0,SCT,2020-07-31
2020-07-31 23:56:00,122228,2020-07-31 23:56:00,19.32,-28.889598,40.349421,229.074545,0.922056,0.426347,Gibbous,792.0,37.0,0.0,37.0,24.0,SCT,2020-07-31
2020-07-31 23:57:00,122229,2020-07-31 23:57:00,19.32,-28.951610,40.567065,228.831172,0.922096,0.426347,Gibbous,792.0,37.0,0.0,37.0,24.0,SCT,2020-07-31


In [213]:
# Define filtering conditions
phases = ['Full', 'Gibbous', 'Crescent', 'Quarter']
alts = [-5, -5, -5, -5]

# Remove data that meets these lunar phase conditions:
for phase, alt in zip(phases, alts):
    df_mo.loc[(df_mo['lunarphaseclass'] == phase) & (df_mo['lunar_alt'] >= alt), 'mags'] = float('nan')

# Filter out cloud coverage
coverages = ['OVC', 'SCT', 'FEW', 'BKN', 'M']
for cover in coverages:
    df_mo.loc[df_mo['CC'] == cover, 'mags'] = float('nan')

In [ ]:

hours = df_mo.index.hour + df_mo.index.minute / 60
hours = np.where(hours < 20, hours + 24, hours)  # Adjust hours to be in the range of 20-30 for 8pm-6am
df_mo['adjusted_hours'] = hours

print(f'{df_mo.index.min()} to {df_mo.index.max()}')

print(df_mo['adjusted_hours'].min(), df_mo['adjusted_hours'].max())


2020-07-01 00:00:00 to 2020-07-31 23:59:00
20.0 43.983333333333334
2020-07-09 01:31:00 to 2020-07-22 03:50:00


/var/folders/hx/hcs55bp10f5fdvcmjf0rmf4r0000gn/T/ipykernel_70673/1060563578.py:3: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



,Unnamed: 0.1,date,mags,sunalt_deg,lunar_alt,lunar_az,lunar_fraction,lunar_phase,lunarphaseclass,Unnamed: 0,cloudcover,cloudcover_low,cloudcover_mid,cloudcover_high,CC,night,adjusted_hours
date,,,,,,,,,,,,,,,,,
2020-07-09 01:31:00,109839,2020-07-09 01:31:00,19.77,-26.453097,-15.439928,288.172375,0.843462,0.645870,Gibbous,242.0,4.0,0.0,4.0,0.0,CLR,2020-07-08,25.516667
2020-07-09 01:32:00,109840,2020-07-09 01:32:00,19.76,-26.431150,-15.212161,287.931269,0.843411,0.645870,Gibbous,242.0,4.0,0.0,4.0,0.0,CLR,2020-07-08,25.533333
2020-07-09 01:33:00,109841,2020-07-09 01:33:00,19.74,-26.408363,-14.984311,287.690191,0.843361,0.645870,Gibbous,242.0,4.0,0.0,4.0,0.0,CLR,2020-07-08,25.550000
2020-07-09 01:34:00,109842,2020-07-09 01:34:00,19.73,-26.384735,-14.756378,287.449058,0.843310,0.645870,Gibbous,242.0,4.0,0.0,4.0,0.0,CLR,2020-07-08,25.566667
2020-07-09 01:35:00,109843,2020-07-09 01:35:00,19.71,-26.360267,-14.528360,287.207925,0.843260,0.645870,Gibbous,242.0,4.0,0.0,4.0,0.0,CLR,2020-07-08,25.583333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-07-22 03:46:00,116971,2020-07-22 03:46:00,18.70,-18.571138,-45.481014,44.403333,0.027297,0.067885,New,556.0,1.0,0.0,1.0,0.0,CLR,2020-07-21,27.766667
2020-07-22 03:47:00,116972,2020-07-22 03:47:00,18.57,-18.451176,-45.703681,44.165700,0.027322,0.067885,New,556.0,1.0,0.0,1.0,0.0,CLR,2020-07-21,27.783333
2020-07-22 03:48:00,116973,2020-07-22 03:48:00,18.43,-18.330671,-45.926288,43.928085,0.027348,0.067885,New,556.0,1.0,0.0,1.0,0.0,CLR,2020-07-21,27.800000


## ERROR IS HAPPENING HERE <br>
TOO MANY DATES BEING REMOVED

In [ ]:

# Filter only nighttime hours (8pm-6am)
t_start = 20 # 8pm
t_end = 6 #  6am
# df_night = df_mo[((hours >= t_start) | (hours < t_end)) & df_mo['mags'].notna()].copy()
df_night = df_mo[(hours >= t_start) & (hours <= t_end + 24) & df_mo['mags'].notna()].copy()
print(f'{df_night.index.min()} to {df_night.index.max()}')

df_night


### Sort valid dates and assign colors

In [215]:

# Filter to a date range 
#   – Check sqm2020_interactive for reasonable data date ranges!
# df_night = df_night[(df_night['night'] >= t_start) & (df_night['night'] <= t_end)]

# Convert 'mags' to candela per square meter
df_night['candela'] = 108000*10**(-0.4*df_night['mags'])

# Get the Timestamp of nights
print(f"First night: {df_night['night'].min()}")
print(f"Last night: {df_night['night'].max()}")
print(f"Number of nights: {len(df_night['night'].unique())}")
nights = sorted(df_night['night'].unique()) # list of sorted nights (Timestamps)

# Assign colors for each night
cmap = cm.brg
colors = cmap(np.linspace(0, 1, len(nights)))
night_color = {night: color for night, color in zip(nights, colors)}

print(f"Number of nights: {len(nights)}")

First night: 2020-07-08 00:00:00
Last night: 2020-07-21 00:00:00
Number of nights: 10
Number of nights: 10


### Plot the data

In [217]:
from numpy.polynomial import polynomial as P

plot_bestfit = False
plot_scatter = True

fig = go.Figure()

for night, group in df_night.groupby('night'):
    
    # Use fractional hours for x-axis, wrapping past-midnight ti000mes
    # hours = group.index.hour + group.index.minute / 60
    # hours = np.where(hours < t_start, hours + 24, hours)  # 1am -> 25, etc

    night_str = night.strftime('%Y-%m-%d')

    print(group)

    if plot_scatter:
        fig.add_trace(
            go.Scatter(
                x=group['adjusted_hours'], 
                y=group['candela'], 
                name=night_str, 
                mode='markers',
                marker=dict(
                    size=8,
                ),
                ))
    
    # Add a best-fit line with variance:
    if plot_bestfit:
        if len(hours) < 6:
            continue

        # Sort by hour for a clean line
        sort_idx = np.argsort(hours)
        hours_sorted = hours[sort_idx]
        mags_sorted = group['candela'][sort_idx]

        # Fit 5th degree polynomial
        coeffs = np.polyfit(hours_sorted, mags_sorted, 4)
        poly = np.poly1d(coeffs)

        # Evaluate on a smooth grid
        x_grid = np.linspace(hours_sorted.min(), hours_sorted.max(), 300)
        y_fit = poly(x_grid)

        # Residual std dev as variance estimate
        y_at_data = poly(hours_sorted)
        residuals = mags_sorted - y_at_data
        std = np.std(residuals)

        color = night_color[night]
        ax.plot(x_grid, y_fit, color=color, linewidth=1.5, label=str(night.date()))
        ax.fill_between(x_grid, y_fit - std, y_fit + std, color=color, alpha=0.15)

fig.update_layout(
    title=dict(
        text=f'SQM Measurements: {loc}, {year}'
    ),
    xaxis=dict(
        title=dict(text='Hour'),
        tickmode='array',
        tickvals=[20,21, 22, 23, 24, 25,26,27,28,29,30],
        ticktext=['8pm','9pm','10pm','11pm','12am','1am','2am','3am','4am','5am','6am']
    ),
    yaxis=dict(
        title=dict(text='Cd/m^2')
    ),
)

fig.show()
fig.write_html(f'./{loc}_{t_start}_{t_end}_interactive.html')

                     Unnamed: 0.1                 date   mags  sunalt_deg  \
date                                                                        
2020-07-09 01:31:00        109839  2020-07-09 01:31:00  19.77  -26.453097   
2020-07-09 01:32:00        109840  2020-07-09 01:32:00  19.76  -26.431150   
2020-07-09 01:33:00        109841  2020-07-09 01:33:00  19.74  -26.408363   
2020-07-09 01:34:00        109842  2020-07-09 01:34:00  19.73  -26.384735   
2020-07-09 01:35:00        109843  2020-07-09 01:35:00  19.71  -26.360267   
2020-07-09 01:36:00        109844  2020-07-09 01:36:00  19.70  -26.334961   
2020-07-09 01:37:00        109845  2020-07-09 01:37:00  19.68  -26.308817   
2020-07-09 01:38:00        109846  2020-07-09 01:38:00  19.66  -26.281837   
2020-07-09 01:39:00        109847  2020-07-09 01:39:00  19.65  -26.254021   
2020-07-09 01:40:00        109848  2020-07-09 01:40:00  19.63  -26.225372   
2020-07-09 01:41:00        109849  2020-07-09 01:41:00  19.60  -26.195889   